# Lecture 4 — Equilibrium and Aiyagari
## 第四讲 —— 均衡与 Aiyagari

**Computational Methods for Heterogeneous-Agent Macro**
**异质性主体宏观的计算方法**

Jeffrey Sun


#### The Aiyagari Model
#### Aiyagari 模型

Like many models, the Aiyagari model consists of three blocks:
1. A household block made of **Stages**
2. A firm side turning capital and labour into goods
3. A notion of equilibrium (steady state)

Familiar economics:
1. A demand side
2. A supply side
3. A notion of equilibrium

与多数模型一样，Aiyagari 模型由三个模块组成：
1. 由 **Stage** 搭成的家庭模块
2. 把资本与劳动转化为商品的企业侧
3. 一个均衡的概念（稳态）

熟悉的经济学：
1. 需求侧
2. 供给侧
3. 均衡的概念


### Environment
### 运行环境


In [1]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using Plots, Random, LinearAlgebra

  Activating project at `~/projects/research/CV is all you need`


# Core Code
# 核心代码


## The Household Block
## 家庭模块


### Parameters
### 参数


In [2]:
u(c) = c <= 0 ? -Inf : log(c)
loggrid(lo, hi, N) = exp.(range(log(lo), log(hi); length = N))

loggrid (generic function with 1 method)

In [3]:
function get_params(;
        β=0.96,
        R=1.04,
        w=1.0,
        b_grid = loggrid(0.05, 200.0, 400),
        w_grid = [2.7, 2.6, 2.5],
        Π      = [0.95 0.04 0.01;
                  0.05 0.9  0.05;
                  0.01 0.04 0.95],
    )

    return params = (;β, R, w,
                    V_shape=(length(b_grid), length(w_grid)),
                    b_grid, w_grid, Π)
end

get_params (generic function with 1 method)

### Utility and Log Grid
### 效用函数与对数网格

In [4]:
"""
Snap a value `x` to the nearest grid index.

把数值 `x` 落到最近的网格下标。
"""
snap_idx(grid, x) = argmin(abs.(grid .- x))

snap_idx

### Backward iteration in stages
### 分阶段后向迭代

The three stages compose into one backward step on a 2-D value function
$V \in \mathbb{R}^{N_b \times N_w}$. Each stage maps one labelled $V$ to the next:

- **Stage 3 (Consumption-Saving) backward.** Grid-search $V^{\mathrm{end}} \mapsto V$.
- **Stage 2 (Income) backward.** Lookup $V \mapsto V^{\mathrm{start}}_{\mathrm{post}}$ at $R\,b^{\mathrm{end}} + w_vals[j]$.
- **Stage 1 (Income Shock) backward.** Matrix-multiply $V^{\mathrm{start}}_{\mathrm{post}} \mapsto V^{\mathrm{start}}_{\mathrm{pre}}$ by $\Pi^\top$.

At the period boundary, $V^{\mathrm{end}} = \beta\, V^{\mathrm{start}}_{\mathrm{pre}}$.

三个阶段合在一起就是 2 维价值函数 $V \in \mathbb{R}^{N_b \times N_w}$ 的一次后向更新。每个阶段把一个 $V$ 映到下一个：

- **第三阶段（后向）。** 网格搜索 $V^{\mathrm{end}} \mapsto V$。
- **第二阶段（后向）。** 在 $R\,b^{\mathrm{end}} + w_vals[j]$ 上查表 $V \mapsto V^{\mathrm{start}}_{\mathrm{post}}$。
- **第一阶段（后向）。** 用 $\Pi^\top$ 做矩阵乘法 $V^{\mathrm{start}}_{\mathrm{post}} \mapsto V^{\mathrm{start}}_{\mathrm{pre}}$。

期界过渡：$V^{\mathrm{end}} = \beta\, V^{\mathrm{start}}_{\mathrm{pre}}$。


In [5]:
"""
Stage 1 (Income Shock) backward: V_start = V_pre_inc * Π'.

第一阶段（后向）：V_start = V_pre_inc * Π'。
"""
income_shock_backward(V_post_inc_shock, params) = V_post_inc_shock * params.Π'

"""
Stage 2 (Income) backward: V_pre_inc(b_end, w) = V(R*b_end + w, w), snapped to nearest grid.
Fully broadcast — no explicit loop.

第二阶段（后向）：V_pre_inc(b_end, w) = V(R*b_end + w, w)，落到最近的网格点。
全部用广播——没有显式循环。
"""
function income_backward(V_post_inc, params)
    (; R, w_grid, b_grid) = params
    b_post_inc = snap_idx.(Ref(b_grid), R .* b_grid .+ w_grid')
    return V_pre_inc = V_post_inc[CartesianIndex.(b_post_inc, axes(b_post_inc, 2)')]
end

"""
Stage 3 (Consumption-Saving) backward: grid-search V_end → V over b_end choices.

第三阶段（消费--储蓄）后向：在 b_end 上做网格搜索，V_end → V。
"""
function consumption_saving_backward(V_end, params)
    # Unpack b_grid
    # 取出 b_grid
    (;b_grid) = params
    
    # Move b_grid to dimension 3 to represent b_end choices
    # 把 b_grid 移到第 3 维，作为 b_end 的候选
    b_next_grid = insertdims(b_grid; dims=(1,2))

    # Move b_grid part of V_end to dimension 3 to represent b_end choices
    # V_end_reshape shape is now [1, w_grid, b_grid]
    # 把 V_end 中对应 b_grid 的轴搬到第 3 维（形状变为 [1, w_grid, b_grid]）
    V_end_reshape = insertdims(permutedims(V_end, (2,1)); dims=1)

    # Maximize along b_end dimension (dim=3)
    # 沿 b_end 维（第 3 维）取最大值
    V = maximum(u.(b_grid .- b_next_grid) .+ V_end_reshape; dims=3)

    # Drop dimension 3 from V and return
    # 去掉第 3 维并返回
    return dropdims(V; dims=3)
end

"""
Read the optimal b_end policy off V_end, by argmax instead of max.

读出最优 b_end 策略：与上式同构，只是把 max 换成 argmax。
"""
function policy_function(V_end, params)
    # Unpack b_grid
    # 取出 b_grid
    (;b_grid) = params
    
    # Move b_grid to dimension 3 to represent b_end choices
    # 把 b_grid 移到第 3 维，作为 b_end 的候选
    b_next_grid = insertdims(b_grid; dims=(1,2))

    # Move b_grid part of V_end to dimension 3 to represent b_end choices
    # V_end_reshape shape is now [1, w_grid, b_grid]
    # 把 V_end 中对应 b_grid 的轴搬到第 3 维（形状变为 [1, w_grid, b_grid]）
    V_end_reshape = insertdims(permutedims(V_end, (2,1)); dims=1)

    # Maximize along b_end dimension (dim=3)
    # 沿 b_end 维（第 3 维）取 argmax
    policy_idxs = argmax(u.(b_grid .- b_next_grid) .+ V_end_reshape; dims=3)

    return [idx[3] for idx in policy_idxs[:,:,1]]
end

policy_function

In [6]:
function bellman_operator(V_end, params)
    (;β) = params
    # Stage 3
    # 第三阶段
    V_post_inc = consumption_saving_backward(V_end, params)

    # Stage 2
    # 第二阶段
    V_post_inc_shock = income_backward(V_post_inc, params)

    # Stage 1
    # 第一阶段
    V_start = income_shock_backward(V_post_inc_shock, params)

    # Passage of time
    # 时间流逝
    V_end_new = β .* V_start
    return V_end_new
end

bellman_operator (generic function with 1 method)

### Value Function Iteration
### 价值函数迭代

Iterate $V \mapsto \mathcal{T}V$ until the max-abs change drops below `tol`.

反复迭代 $V \mapsto \mathcal{T}V$，直到最大绝对变化小于 `tol`。


In [7]:
function solve_vfi(params; tol = 1e-6, maxiter = 2000, verbosity = 0)
    V = zeros(params.V_shape)

    Δ, iters = Inf, 0
    while Δ >= tol
        TV = bellman_operator(V, params)
        Δ  = maximum(abs.(TV .- V))
        V  = TV
        iters += 1
        iters > maxiter && error("VFI did not converge in $maxiter iterations")
    end

    verbosity >= 1 && println("VFI converged in $iters iterations with error $Δ")
    return V
end

solve_vfi (generic function with 1 method)

In [8]:
params = get_params()
V = solve_vfi(params; verbosity=1)

VFI converged in 356 iterations with error 9.84368959677795e-7


400×3 Matrix{Float64}:
 23.6723  23.2872  22.8984
 23.6723  23.2872  22.8984
 23.6723  23.2872  22.8984
 23.6723  23.2872  22.8984
 23.6723  23.2872  22.8984
 23.6723  23.2872  22.8984
 23.6723  23.2872  22.8984
 23.6723  23.2872  22.8984
 23.6723  23.2872  22.8984
 23.6723  23.2872  22.8984
  ⋮                
 57.7003  57.7003  57.7003
 58.1111  58.1111  58.1111
 58.5182  58.5182  58.5182
 58.9215  58.9215  58.9215
 59.3208  59.3208  59.3208
 59.6664  59.596   59.3358
 59.6692  59.6103  59.6077
 59.6692  59.6103  59.6077
 59.6692  59.6103  59.6077

### Forward iteration in stages
### 分阶段前向迭代

We can simulate each population matrix forward though each stage, one at a time.

- **Stage 1 (Income Shock) forward.** $\lambda \mapsto \lambda\, \Pi$.
- **Stage 2 (Income) forward.** Each $(b^{\mathrm{end}}, w)$ cell moves to $(R\,b^{\mathrm{end}} + w_vals[j],\, w)$.
- **Stage 3 (Consumption-Saving) forward.** Each $(b, w)$ cell moves to $(b - c^\star(b, w),\, w)$.

Both wealth re-bins snap to the nearest grid point.

三个阶段也可以合成 2 维分布 $\lambda \in \mathbb{R}^{N_b \times N_w}$ 的一次*前向*更新：

- **第一阶段（前向）。** $\lambda \mapsto \lambda\, \Pi$。
- **第二阶段（前向）。** 每个 $(b^{\mathrm{end}}, w)$ 单元移到 $(R\,b^{\mathrm{end}} + w_vals[j],\, w)$。
- **第三阶段（前向）。** 每个 $(b, w)$ 单元移到 $(b - c^\star(b, w),\, w)$。

两次沿财富维的重排都落到最近的网格点。


In [9]:
"""
Stage 1 forward: λ_post = λ_pre * Π.

第一阶段（前向）：λ_post = λ_pre * Π。
"""
income_shock_forward(λ, Π) = λ * Π

"""
Stage 2 forward: each (i_b_end, i_w) cell moves to (snap(R*b_end + w), i_w).
Vectorize the destination index via broadcasting; one tight scatter.

第二阶段（前向）：每个 (i_b_end, i_w) 单元移到 (snap(R*b_end + w), i_w)。
目标下标用广播算出；一次紧凑散布。
"""

function change_λ_wealth(λ, b_inds_new)
    λ_new = zero(λ)
    for old_idx in CartesianIndices(λ)
        λ_new[b_inds_new[old_idx], old_idx[2]] += λ[old_idx]
    end
    return λ_new
end

function income_forward(λ, params)
    (; R, w_grid, b_grid) = params
    dest = snap_idx.(Ref(b_grid), R .* b_grid .+ w_grid')               # (N_b, N_w)
    return change_λ_wealth(λ, dest)
end

"""
Stage 3 forward: each (i_b, i_w) cell moves to (c_ind[i_b, i_w], i_w).
c_ind is exactly the destination index — direct scatter.

第三阶段（前向）：每个 (i_b, i_w) 单元移到 (c_ind[i_b, i_w], i_w)。
c_ind 本身就是目标下标——直接散布。
"""
consumption_saving_forward(λ, b_inds_new, params) = change_λ_wealth(λ, b_inds_new)

"""
One forward step of the composite operator T*:
λ → income shock → income → consumption saving.

复合算子 T* 的一次前向更新：λ → 收入冲击 → 收入 → 消费储蓄。
"""
function simulate_population_forward(λ, b_inds_new, params)
    # Stage 1
    # 第一阶段
    λ = income_shock_forward(λ, params.Π)

    # Stage 2
    # 第二阶段
    λ = income_forward(λ, params)

    # Stage 3
    # 第三阶段
    λ = consumption_saving_forward(λ, b_inds_new, params)
    
    return λ
end


simulate_population_forward

In [10]:
function find_steady_state_population(V_end, params; tol = 1e-5, maxiter = 10_000, verbosity = 0)

    b_inds_new = policy_function(V_end, params)
    
    λ = fill(1/length(params.V_shape), params.V_shape) # Start with households evenly distributed across gridpoints

    Δ, iters = Inf, 0
    while Δ >= tol
        λ_new = simulate_population_forward(λ, b_inds_new, params)
        Δ = maximum(abs.(λ_new .- λ))
        λ = λ_new
        iters += 1
        iters > maxiter && error("Steady state λ did not converge in $maxiter iterations")
    end

    verbosity >= 1 && println("Steady state λ converged in $iters iterations with error $Δ")
    return λ
end

find_steady_state_population (generic function with 1 method)

In [23]:
function solve_aiyagari_steady_state(params; verbosity=0)
    V_end = solve_vfi(params; verbosity)
    λ = find_steady_state_population(V_end, params; verbosity)
    return (;V_end, λ)
end

function solve_aiyagari_steady_state(;verbosity=0, param_vals...)
    return solve_aiyagari_steady_state(get_params(;param_vals...); verbosity)
end

solve_aiyagari_steady_state (generic function with 2 methods)

In [12]:
params = get_params(;R=1.04)
(;V_end, λ) = solve_aiyagari_steady_state(params; verbosity=0)

(V_end = [23.67232910939439 23.287179584922146 22.89843850673867; 23.67232910939439 23.287179584922146 22.89843850673867; … ; 59.669214659916896 59.61034198828938 59.60772542510592; 59.669214659916896 59.61034198828938 59.60772542510592], λ = [0.0 0.0 0.0; 0.0 0.0 0.0; … ; 0.0 0.0 0.0; 0.0 0.0 0.0])

In [13]:
sum(λ .* params.b_grid) # Aggregate capital in the economy

18770.083639402765

In [14]:
params.b_grid

400-element Vector{Float64}:
   0.05000000000000001
   0.05105023241317379
   0.05212252458878116
   0.05321733988437287
   0.05433515139016193
   0.05547644213345498
   0.056641705287377234
   0.05783144438398204
   0.05904617353183631
   0.06028641763817668
   ⋮
 169.35898470386456
 172.9163106078285
 176.54835689116376
 180.2566930291573
 184.04292146337195
 187.90867829409248
 191.8556339873144
 195.88549409658413
 199.99999999999991

## Computing Moments
## 计算总量

### Cross-sectional aggregates from $V$ and $c^\star$
### 从 $V$、$c^\star$ 直接读出横截面总量

Aggregate welfare $\bar V$, consumption $\bar C$, wealth $\bar K$ at each $t$ — all inner products against the marginal wealth distribution.

总福利 $\bar V$、总消费 $\bar C$、总财富 $\bar K$——都是与边际财富分布的内积。


In [15]:
get_c_policy(b_grid, b_grid_policy) = b_grid .- b_grid[b_grid_policy]

function compute_household_aggregates(;param_vals...)
    params = get_params(;param_vals...)
    (V_end, λ) = solve_aiyagari_steady_state(params)

    
    b_grid_policy = policy_function(V_end, params)
    c_policy = get_c_policy(params.b_grid, b_grid_policy)
    
    L_agg = sum(λ)
    K_agg = sum(params.b_grid .* λ)
    V_agg = sum(V_end .* λ)
    C_agg = sum(c_policy .* λ)

    return (;V_end, λ, L_agg, K_agg, V_agg, C_agg)
end

compute_household_aggregates (generic function with 1 method)

In [16]:
params = get_params(;R=1.04)
(V_end, λ) = solve_aiyagari_steady_state(params)

@show V_agg = sum(V_end .* λ)
@show K_agg = sum(params.b_grid .* λ)

b_grid_policy = policy_function(V_end, params)
c_grid_policy = get_c_policy(params.b_grid, b_grid_policy)

@show C_agg = sum(c_grid_policy .* λ)

@show L_agg = sum(λ)


VFI converged in 356 iterations with error 9.84368959677795e-7
Steady state λ converged in 1141 iterations with error 9.986706672331569e-6
V_agg = sum(V_end .* λ) = 19016.717558002063
K_agg = sum(params.b_grid .* λ) = 18770.083639402765
C_agg = sum(c_grid_policy .* λ) = 1896.7521126957877
L_agg = sum(λ) = 599.9999999999911


599.9999999999911

In [17]:
compute_Y(K, L; α=1/3) = K^α * L^(1-α)

compute_Y (generic function with 1 method)

In [18]:
res = compute_household_aggregates(;R=1.03)

VFI converged in 347 iterations with error 9.91283897633366e-7
Steady state λ converged in 2181 iterations with error 9.972564981808318e-6


(V_end = [23.518406335437152 23.158873934425813 22.79692615242636; 23.518406335437152 23.158873934425813 22.79692615242636; … ; 52.81417713675179 52.70696167011926 52.61662992417005; 52.81417713675179 52.70696167011926 52.61662992417005], λ = [0.0 0.0 0.0; 0.0 0.0 0.0; … ; 0.0 0.0 0.0; 0.0 0.0 0.0], L_agg = 599.9999999999784, K_agg = 11370.009529276362, V_agg = 16648.59858157776, C_agg = 839.9471938724707)

In [19]:
Y_agg = compute_Y(res.K_agg, res.L_agg)

1599.635780447823

In [20]:
res.C_agg

839.9471938724707

In [21]:
function compute_excess_demand(R)
    res = compute_household_aggregates(;R)
    
    Y_agg = compute_Y(res.K_agg, res.L_agg)

    return excess_demand = (res.C_agg - Y_agg)/Y_agg
end

compute_excess_demand (generic function with 1 method)

In [22]:
compute_excess_demand(1.05)

VFI converged in 356 iterations with error 9.8408977322606e-7
Steady state λ converged in 3176 iterations with error 9.990166205398054e-6


0.15096070407050316

In [33]:
tol = 0.01
lr = 0.001

R = 1.01
err = Inf
while err > tol
    excess_demand = compute_excess_demand(R)
    
    R_new = R - excess_demand*lr
    err = abs(excess_demand)

    R = R_new
    
    println("R=$R\t excess_demand=$excess_demand")
end

println("Equilibrium R: $R")

R=1.0109321576475578	 excess_demand=-0.9321576475577228
R=1.0118643152951157	 excess_demand=-0.9321576475577228
R=1.0127964729426735	 excess_demand=-0.9321576475577228
R=1.0137286305902313	 excess_demand=-0.9321576475577228
R=1.0146607882377892	 excess_demand=-0.9321576475577228
R=1.015592945885347	 excess_demand=-0.9321576475577228
R=1.0165251035329048	 excess_demand=-0.9321576475577228
R=1.0174572611804626	 excess_demand=-0.9321576475577228
R=1.018389418828019	 excess_demand=-0.9321576475562137
R=1.019321576475384	 excess_demand=-0.9321576473651716
R=1.0202207188745884	 excess_demand=-0.8991423992043716
R=1.0211163744765481	 excess_demand=-0.895655601959657
R=1.022012037326975	 excess_demand=-0.8956628504268269
R=1.0226436898211906	 excess_demand=-0.6316524942155441
R=1.023270601828399	 excess_demand=-0.6269120072083295
R=1.0238387582875557	 excess_demand=-0.5681564591566658
R=1.0246336725210177	 excess_demand=-0.7949142334619326
R=1.0253531800318731	 excess_demand=-0.719507510855383